# Datu tīrīšana: pirms un pēc

Vispirms apskatīsim, ko dators redz **sākotnējā, netīrītā datu kopā**.

Pēc tam iztīrīsim datu kopu ar `Data_cleaning_task.ipynb` un salīdzināsim tos pašus piemērus **pirms un pēc tīrīšanas**.

> Cilvēkam "Jā" un "Jā " var šķist viena un tā pati vērtība, bet datoram tās ir divas atšķirīgas vērtības.

# 1. Ielādējam sākotnējo datu kopu

Izvēlieties failu **`Sintetisko-datu-kopa.xlsx`**.

In [ ]:
#@title Augšupielādēt sākotnējo datu kopu
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from io import BytesIO
import base64
import re

uploaded = files.upload()
raw_filename = next(iter(uploaded))

raw_file = pd.read_excel(raw_filename, header=None)

headers = (
    ["STUDENTA ID"] +
    [str(x).replace("\n", " ").strip() for x in raw_file.iloc[1, 1:].tolist()]
)

df_raw = raw_file.iloc[2:].copy()
df_raw.columns = headers
df_raw = df_raw.reset_index(drop=True)

print(f"Sākotnējā datu kopa: {len(df_raw)} rindas, {len(df_raw.columns)} kolonnas.")

# PIRMS TĪRĪŠANAS

Zemāk visi piemēri tiek veidoti tikai no **sākotnējās datu kopas**.

## 2. `Jā / Nē`

Kolonnā **“Vai ir studējusi ārzemēs?”** bija paredzētas tikai divas atbildes: `Jā` un `Nē`.

Apskatīsim, cik dažādas kategorijas dators patiesībā redz.

In [ ]:
#@title Jā / Nē — pirms tīrīšanas
col_abroad_raw = [c for c in df_raw.columns if c.startswith("Vai ir studējusi ārzemēs?")][0]

def plain_label(value):
    if pd.isna(value):
        return "(trūkst)"
    return str(value)

raw_abroad_counts = df_raw[col_abroad_raw].value_counts(dropna=False)

plt.figure(figsize=(9, 5))
plt.barh(
    [plain_label(x) for x in raw_abroad_counts.index],
    raw_abroad_counts.values
)
plt.title("Vai ir studējusi ārzemēs? — pirms tīrīšanas")
plt.xlabel("Ierakstu skaits")
plt.ylabel("Datorā saglabātā vērtība")
plt.tight_layout()
plt.show()

print(f"Dators redz {len(raw_abroad_counts)} dažādas kategorijas (ieskaitot trūkstošās vērtības).")

**Pievērsiet uzmanību:** grafikā dažas kategorijas var izskatīties pilnīgi vienādi, piemēram, `Jā` un `Jā `, lai gan dators tās saskaita atsevišķi.


## 3. Dzimšanas gadi

Dzimšanas datumi ir saglabāti dažādos formātos.

Šeit izmantosim ļoti vienkāršu pieeju: mēģināsim no teksta atrast četrciparu gadu.

In [ ]:
#@title Dzimšanas gadi — pirms tīrīšanas
col_date_raw = [c for c in df_raw.columns if c.startswith("Dzimšanas datums")][0]

def naive_year(value):
    if not isinstance(value, str):
        return np.nan
    m = re.search(r"(\d{4})", value)
    return float(m.group(1)) if m else np.nan

raw_years = df_raw[col_date_raw].apply(naive_year)

raw_years_plot = raw_years[(raw_years >= 1900) & (raw_years <= 2010)]
bins_years = np.arange(1900, 2016, 5)

plt.figure(figsize=(9, 5))
plt.hist(raw_years_plot.dropna(), bins=bins_years)
plt.title("Dzimšanas gadi — pirms tīrīšanas")
plt.xlabel("Dzimšanas gads")
plt.ylabel("Ierakstu skaits")
plt.xlim(1900, 2010)
plt.tight_layout()
plt.show()

print(f"Aizpildīti dzimšanas datumi: {df_raw[col_date_raw].notna().sum()}")
print(f"Ar šo vienkāršo metodi gadu izdevās nolasīt: {raw_years.notna().sum()}")

## 4. Apmierinātība ar dzīvi

Instrukcijā bija paredzētas veselas vērtības no **1 līdz 5**.

Apskatīsim, kādas vērtības faktiski ir datu kopā.

In [ ]:
#@title Apmierinātība — pirms tīrīšanas
col_sat_raw = [c for c in df_raw.columns if c.startswith("Apmierinātība ar dzīvi")][0]

raw_sat_labels = df_raw[col_sat_raw].astype("string").fillna("(trūkst)")
raw_sat_counts = raw_sat_labels.value_counts()

plt.figure(figsize=(10, 5))
plt.bar(raw_sat_counts.index.astype(str), raw_sat_counts.values)
plt.title("Apmierinātība ar dzīvi — pirms tīrīšanas")
plt.xlabel("Vērtība")
plt.ylabel("Ierakstu skaits")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("Unikālās vērtības:")
print(list(raw_sat_counts.index))

## 5. Vēlamie ikmēneša ienākumi

Daļa vērtību ir skaitļi, bet daļā skaitlim ir pievienots teksts, piemēram, `5000 eiro`, `1876(neto)` vai `~3000`.

Apskatīsim, cik ierakstu var uzreiz izmantot kā skaitļus.

In [ ]:
#@title Ienākumi — pirms tīrīšanas
col_income_raw = [c for c in df_raw.columns if c.startswith("Vēlamie ikmēneša ienākumi")][0]

raw_income = pd.to_numeric(df_raw[col_income_raw], errors="coerce")
raw_income_plot = raw_income[(raw_income >= 0) & (raw_income <= 10000)]
bins_income = np.linspace(0, 10000, 11)

plt.figure(figsize=(9, 5))
plt.hist(raw_income_plot.dropna(), bins=bins_income)
plt.title("Vēlamie ikmēneša ienākumi — pirms tīrīšanas")
plt.xlabel("EUR mēnesī")
plt.ylabel("Ierakstu skaits")
plt.xlim(0, 10000)
plt.tight_layout()
plt.show()

print(f"Aizpildīti ienākumu ieraksti: {df_raw[col_income_raw].notna().sum()}")
print(f"Uzreiz kā skaitļus izdevās nolasīt: {raw_income.notna().sum()}")

# Tagad iztīrām datu kopu

Atveriet **`Data_cleaning_task.ipynb`**, izejiet tam cauri un beigās lejupielādējiet failu:

**`Sintetisko-datu-kopa_CLEANED.xlsx`**

Pēc tam atgriezieties šeit.

# 6. Ielādējam iztīrīto datu kopu

In [ ]:
#@title Augšupielādēt iztīrīto datu kopu
uploaded_clean = files.upload()
clean_filename = next(iter(uploaded_clean))

df_clean = pd.read_excel(clean_filename)

print(f"Iztīrītā datu kopa: {len(df_clean)} rindas, {len(df_clean.columns)} kolonnas.")

# PIRMS UN PĒC

Tagad atkārtosim tos pašus piemērus un katrā blokā parādīsim **divus grafikus blakus**:

**PIRMS TĪRĪŠANAS** | **PĒC TĪRĪŠANAS**

In [ ]:
#@title Tehniskā sagatavošana salīdzinājumiem
def fig_to_data_uri(fig):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=130, bbox_inches="tight")
    plt.close(fig)
    buffer.seek(0)
    encoded = base64.b64encode(buffer.read()).decode("ascii")
    return "data:image/png;base64," + encoded

def show_pair(fig_left, fig_right, left_note="", right_note=""):
    left_uri = fig_to_data_uri(fig_left)
    right_uri = fig_to_data_uri(fig_right)

    html = f"""
    <div style="display:flex; gap:18px; align-items:flex-start; width:100%;">
      <div style="width:50%;">
        <div style="font-weight:700; font-size:17px; margin-bottom:6px;">PIRMS TĪRĪŠANAS</div>
        <img src="{left_uri}" style="width:100%; height:auto;">
        <div style="margin-top:6px; font-size:14px;">{left_note}</div>
      </div>
      <div style="width:50%;">
        <div style="font-weight:700; font-size:17px; margin-bottom:6px;">PĒC TĪRĪŠANAS</div>
        <img src="{right_uri}" style="width:100%; height:auto;">
        <div style="margin-top:6px; font-size:14px;">{right_note}</div>
      </div>
    </div>
    """
    display(HTML(html))

## 7. `Jā / Nē` — pirms un pēc

Tagad var redzēt, ka pirms tīrīšanas dažas atsevišķi saskaitītas kategorijas vizuāli izskatījās vienādi, bet pēc standartizācijas tās ir apvienotas.


In [ ]:
#@title Jā / Nē — salīdzinājums
col_abroad_clean = [c for c in df_clean.columns if c.startswith("Vai ir studējusi ārzemēs?")][0]

before = df_raw[col_abroad_raw].value_counts(dropna=False)
after = df_clean[col_abroad_clean].value_counts(dropna=False)

xmax = max(before.max(), after.max()) * 1.10

fig1 = plt.figure(figsize=(6.5, 4.8))
ax1 = fig1.add_axes([0.22, 0.14, 0.72, 0.76])
ax1.barh([plain_label(x) for x in before.index], before.values)
ax1.set_title("Kategorijas sākotnējos datos")
ax1.set_xlabel("Ierakstu skaits")
ax1.set_xlim(0, xmax)

fig2 = plt.figure(figsize=(6.5, 4.8))
ax2 = fig2.add_axes([0.22, 0.14, 0.72, 0.76])
ax2.barh([plain_label(x) for x in after.index], after.values)
ax2.set_title("Kategorijas pēc standartizācijas")
ax2.set_xlabel("Ierakstu skaits")
ax2.set_xlim(0, xmax)

show_pair(
    fig1,
    fig2,
    left_note=f"Kategorijas: {len(before)}",
    right_note=f"Kategorijas: {len(after)}"
)

## 8. Dzimšanas gadi — pirms un pēc

In [ ]:
#@title Dzimšanas gadi — salīdzinājums
col_date_clean = [c for c in df_clean.columns if c.startswith("Dzimšanas datums") and "original" not in c.lower()][0]

clean_date_text = df_clean[col_date_clean].astype("string").str.strip()

# Pēc tīrīšanas datuma formāts var būt atšķirīgs
# (piemēram, dd.mm.yyyy, yyyy-mm-dd, dd/mm/yyyy vai mm/dd/yyyy).
# Šajā vizualizācijā mums nepieciešams tikai gads, tāpēc atrodam četrciparu gadu
# neatkarīgi no izvēlētā datuma formāta.
clean_years = pd.to_numeric(
    clean_date_text.str.extract(r"(\d{4})", expand=False),
    errors="coerce"
)

clean_years_plot = clean_years[(clean_years >= 1900) & (clean_years <= 2010)]

raw_hist, _ = np.histogram(raw_years_plot.dropna(), bins=bins_years)
clean_hist, _ = np.histogram(clean_years_plot.dropna(), bins=bins_years)
ymax = max(raw_hist.max(), clean_hist.max()) * 1.15

fig1 = plt.figure(figsize=(6.5, 4.6))
ax1 = fig1.add_axes([0.12, 0.15, 0.83, 0.76])
ax1.hist(raw_years_plot.dropna(), bins=bins_years)
ax1.set_title("Vienkārši nolasītie gadi")
ax1.set_xlabel("Dzimšanas gads")
ax1.set_ylabel("Ierakstu skaits")
ax1.set_xlim(1900, 2010)
ax1.set_ylim(0, ymax)

fig2 = plt.figure(figsize=(6.5, 4.6))
ax2 = fig2.add_axes([0.12, 0.15, 0.83, 0.76])
ax2.hist(clean_years_plot.dropna(), bins=bins_years)
ax2.set_title("Gadi pēc datumu standartizācijas")
ax2.set_xlabel("Dzimšanas gads")
ax2.set_ylabel("Ierakstu skaits")
ax2.set_xlim(1900, 2010)
ax2.set_ylim(0, ymax)

show_pair(
    fig1,
    fig2,
    left_note=f"Nolasīti gadi: {raw_years.notna().sum()}",
    right_note=f"Nolasāmi datumi: {clean_years.notna().sum()}"
)

## 9. Apmierinātība ar dzīvi — pirms un pēc

In [ ]:
#@title Apmierinātība — salīdzinājums
col_sat_clean = [c for c in df_clean.columns if c.startswith("Apmierinātība ar dzīvi")][0]

before = raw_sat_counts

clean_sat_numeric = pd.to_numeric(
    df_clean[col_sat_clean].astype("string").str.replace(",", ".", regex=False),
    errors="coerce"
)
valid_mask = clean_sat_numeric.isin([1, 2, 3, 4, 5])
after = clean_sat_numeric[valid_mask].value_counts().sort_index()

ymax = max(before.max(), after.max()) * 1.10

fig1 = plt.figure(figsize=(6.5, 4.8))
ax1 = fig1.add_axes([0.14, 0.24, 0.81, 0.67])
ax1.bar(before.index.astype(str), before.values)
ax1.set_title("Visas sākotnējās vērtības")
ax1.set_xlabel("Vērtība")
ax1.set_ylabel("Ierakstu skaits")
ax1.set_ylim(0, ymax)
ax1.tick_params(axis="x", rotation=50)

fig2 = plt.figure(figsize=(6.5, 4.8))
ax2 = fig2.add_axes([0.14, 0.18, 0.81, 0.73])
ax2.bar(after.index.astype(int).astype(str), after.values)
ax2.set_title("Tikai derīgās vērtības 1–5")
ax2.set_xlabel("Vērtība")
ax2.set_ylabel("Ierakstu skaits")
ax2.set_ylim(0, ymax)

invalid_count = int((~valid_mask & df_clean[col_sat_clean].notna()).sum())

show_pair(
    fig1,
    fig2,
    left_note=f"Sākotnējās kategorijas: {len(before)}",
    right_note=f"Manuālai pārbaudei: {invalid_count}"
)

## 10. Ienākumi — pirms un pēc

In [ ]:
#@title Ienākumi — salīdzinājums
if "income_eur" in df_clean.columns:
    clean_income = pd.to_numeric(df_clean["income_eur"], errors="coerce")
else:
    col_income_clean = [c for c in df_clean.columns if c.startswith("Vēlamie ikmēneša ienākumi")][0]
    clean_income = pd.to_numeric(df_clean[col_income_clean], errors="coerce")

clean_income_plot = clean_income[(clean_income >= 0) & (clean_income <= 10000)]

raw_hist, _ = np.histogram(raw_income_plot.dropna(), bins=bins_income)
clean_hist, _ = np.histogram(clean_income_plot.dropna(), bins=bins_income)
ymax = max(raw_hist.max(), clean_hist.max()) * 1.15

fig1 = plt.figure(figsize=(6.5, 4.6))
ax1 = fig1.add_axes([0.12, 0.15, 0.83, 0.76])
ax1.hist(raw_income_plot.dropna(), bins=bins_income)
ax1.set_title("Skaitliski nolasītie ienākumi")
ax1.set_xlabel("EUR mēnesī")
ax1.set_ylabel("Ierakstu skaits")
ax1.set_xlim(0, 10000)
ax1.set_ylim(0, ymax)

fig2 = plt.figure(figsize=(6.5, 4.6))
ax2 = fig2.add_axes([0.12, 0.15, 0.83, 0.76])
ax2.hist(clean_income_plot.dropna(), bins=bins_income)
ax2.set_title("Ienākumi pēc strukturēšanas")
ax2.set_xlabel("EUR mēnesī")
ax2.set_ylabel("Ierakstu skaits")
ax2.set_xlim(0, 10000)
ax2.set_ylim(0, ymax)

show_pair(
    fig1,
    fig2,
    left_note=f"Skaitliski izmantojami ieraksti: {raw_income.notna().sum()}",
    right_note=f"Skaitliski izmantojami ieraksti: {clean_income.notna().sum()}"
)

# Noslēguma jautājumi

Atbildiet uz noslēguma jautājumiem, kas atrodami `Data_cleaning_task.ipynb` un `Uzdevuma_nosacijumi.docx` failos.